In [1]:
import sys

import numpy as np
import pandas as pd
import scipy.stats as ss
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl

In [2]:
from legnet_embedding import LegNetEmbedding
sys.path.append("../../predictor/model/")
import utrdata_cl as utrdata
from pl_regressor import RNARegressor

## Loading data

In [3]:
utr_type = "utr5"

In [4]:
if utr_type == "utr5":
    MODEL_PATH = "../../predictor/regression_multiple/saved_models/model-utr5-deltas-epoch=9-step=840.ckpt"
elif utr_type == "utr3":
    MODEL_PATH = "../../predictor/regression_multiple/saved_models/model-utr3-deltas-epoch=9-step=1330.ckpt"
else:
    raise ValueError

In [5]:
def generate_random_sequence(length, alphabet=np.array(list("ACGT"))):
    percs = np.random.dirichlet(alpha=np.full(fill_value=1.0, shape=(4,)))
    seq = np.random.choice(alphabet, size=(length,), replace=True, p=percs)
    return "".join(seq)

In [6]:
if utr_type == "utr5":
    utr_length = 50
    cell_types = ["c1", "c2", "c4", "c6", "c17"]
elif utr_type == "utr3":
    utr_length = 240
    cell_types = ["c1", "c2", "c4", "c6", "c13", "c17"]

In [7]:
from itertools import product

n_seqs = 1_000_000
np.random.seed(777)
df = pd.DataFrame(product([generate_random_sequence(utr_length) for _ in range(n_seqs)], cell_types), columns=["seq", "cell_type"])
df.head()

,seq,cell_type
0,TTTTCTAATCTTTTCCCCCCTTTCCTCTTTTATTTACTTGTCTTGT...,c1
1,TTTTCTAATCTTTTCCCCCCTTTCCTCTTTTATTTACTTGTCTTGT...,c2
2,TTTTCTAATCTTTTCCCCCCTTTCCTCTTTTATTTACTTGTCTTGT...,c4
3,TTTTCTAATCTTTTCCCCCCTTTCCTCTTTTATTTACTTGTCTTGT...,c6
4,TTTTCTAATCTTTTCCCCCCTTTCCTCTTTTATTTACTTGTCTTGT...,c17


In [8]:
ct_classes = df["cell_type"].unique()
num_classes = ct_classes.shape[0]
num_classes

5

In [9]:
batch_size = 1024

In [10]:
num_workers = 32

In [11]:
test_set = utrdata.UTRData(
    df=df,
    predict_cols=[],
    features=("sequence", "positional", "conditions"),
    construct_type=utr_type,
    augment=False,
    augment_test_time=False,
    augment_kws=dict(
        extend_left=0,
        extend_right=0,
        shift_left=0,
        shift_right=0,
        revcomp=False,
    ),
)

In [12]:
# Creating DataLoaders
dl_test = DataLoader(
    test_set,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    drop_last=False
)

In [13]:
ckpt = torch.load(MODEL_PATH)
ckpt["hyper_parameters"]["model_class"] = LegNetEmbedding
ckpt["hyper_parameters"]["model_class"]

loaded_model = RNARegressor(**ckpt["hyper_parameters"])
loaded_model.load_state_dict(ckpt["state_dict"])

<All keys matched successfully>

In [14]:
progressbar_callback = pl.callbacks.TQDMProgressBar(refresh_rate=0.5)
trainer = pl.Trainer(
    callbacks=[progressbar_callback],
    logger=False,
    accelerator="gpu",
    devices=[1],
    deterministic=True,
)

prediction = trainer.predict(model=loaded_model, dataloaders=dl_test)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Traceback (most recent call last):
  File "/home/arsen_l/.miniconda3/envs/ml/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/arsen_l/.miniconda3/envs/ml/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/arsen_l/.miniconda3/envs/ml/lib/python3.11/multipr

Predicting: 0it [00:00, ?it/s]

In [15]:
pred_tuples, _ = zip(*prediction)
embed, pred = zip(*pred_tuples)
embed = torch.concat(embed).numpy()
pred = torch.concat(pred).numpy()

In [16]:
seq_embedding = embed.reshape(-1, num_classes * embed.shape[-1], order="C")

In [17]:
pred_mass_center = pred[:, 1].reshape(-1, num_classes)

In [18]:
result = pd.DataFrame(
    pred_mass_center,
    columns=pd.MultiIndex.from_product([["pred_mass_center"], cell_types]),
    index=df.iloc[::num_classes]["seq"]
)
result["pred_mass_center"].head()

,c1,c2,c4,c6,c17
seq,,,,,
TTTTCTAATCTTTTCCCCCCTTTCCTCTTTTATTTACTTGTCTTGTATCC,2.498896,2.572722,2.478604,2.371122,2.511424
ATTTTAATTTTTTTATTTGGTTTGATTATTAGTTTTATTGTTTTGATTTT,2.418928,2.449709,2.321141,2.233561,2.440281
CCTCCCCCTCGTATTTCTCCTCCCTAGGTCTTCCTGTGTCGTCCCCACGC,2.431649,2.530180,2.470930,2.411971,2.449337
AAGAATAATAAGAGAGGGGGTAAAAATATTAAGAGGACGAACATAAAAAG,2.356107,2.516393,2.374801,2.288465,2.367180
CGCGAAGAAATGCGTCGAGGTGAGAGTGTACGCTGACGCACAAGCAACAG,2.471555,2.446281,2.390172,2.416741,2.346086


In [19]:
seq_embedding_df = pd.DataFrame(
    seq_embedding,
    columns=pd.MultiIndex.from_product([
        ["embedding"],
        [f"{ct}_{i:03d}" for ct in result["pred_mass_center"].columns for i in range(embed.shape[-1])]
    ]),
    index=result.index)
result = pd.concat([result, seq_embedding_df], axis=1)

In [20]:
result["embedding"]

,c1_000,c1_001,c1_002,c1_003,c1_004,c1_005,c1_006,c1_007,c1_008,c1_009,...,c17_022,c17_023,c17_024,c17_025,c17_026,c17_027,c17_028,c17_029,c17_030,c17_031
seq,,,,,,,,,,,,,,,,,,,,,
TTTTCTAATCTTTTCCCCCCTTTCCTCTTTTATTTACTTGTCTTGTATCC,0.170352,0.137776,0.149795,0.115654,0.163539,0.139428,0.096351,0.192964,0.142154,0.103248,...,0.006233,0.099934,0.137141,0.113065,0.171329,0.181166,0.105965,0.104364,0.197847,0.123752
ATTTTAATTTTTTTATTTGGTTTGATTATTAGTTTTATTGTTTTGATTTT,0.173191,0.147516,0.149320,0.077798,0.177750,0.119023,0.067370,0.253474,0.125348,0.111351,...,-0.057611,0.163677,0.117372,0.095032,0.228335,0.177238,0.114249,0.186913,0.227315,0.110784
CCTCCCCCTCGTATTTCTCCTCCCTAGGTCTTCCTGTGTCGTCCCCACGC,0.145683,0.095493,0.161666,0.082157,0.158707,0.120799,0.112593,0.135415,0.155509,0.146323,...,-0.022045,0.141391,0.124441,0.148633,0.157695,0.038647,0.187118,0.046599,0.204002,0.148705
AAGAATAATAAGAGAGGGGGTAAAAATATTAAGAGGACGAACATAAAAAG,0.126414,0.144119,0.099966,0.129991,0.160339,0.038473,0.049844,0.158385,0.095619,0.052251,...,0.075361,0.097821,0.092315,0.208875,0.095823,-0.038640,0.187808,0.162919,0.256677,0.089351
CGCGAAGAAATGCGTCGAGGTGAGAGTGTACGCTGACGCACAAGCAACAG,0.119197,0.146393,0.106693,0.139165,0.116917,0.085925,0.114624,0.118348,0.139122,0.097442,...,0.084804,0.087071,0.110562,0.155491,0.142451,0.023213,0.132662,0.097926,0.179296,0.158784
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GCGATGAGGGGGGGGAGGGGGGGTGGAGGGGTGCTGGCGGTAGGGGGGGA,0.151594,0.100273,0.158391,0.132210,0.120637,0.084708,0.141291,0.221139,0.143033,0.140452,...,-0.161854,0.220300,0.088223,0.136172,0.200903,0.037715,0.148659,0.098186,0.076805,0.031262
ATTTATATATGTATATAAAAAATTGAGATATAACAAAGGTGCATTTAATT,0.165386,0.122378,0.130052,0.052082,0.164876,0.098828,0.063869,0.108468,0.059540,0.042516,...,0.005841,0.078926,0.093939,0.183158,0.176670,-0.040088,0.181642,0.184352,0.225052,0.105895
GTAGCTGGAGAGCTTGTGGGAGAGGATGATGCTGGTGCTTGTTTTGGGAG,0.118994,0.074683,0.135026,0.098908,0.083165,0.093765,0.125104,0.127546,0.103417,0.091937,...,-0.033034,0.110221,0.098632,0.102976,0.115699,0.025634,0.117613,0.087397,0.124531,0.096928


### Counting k-mers

In [21]:
sys.path.append("../../benchmark/kmers")
from kmer_model import KmerLinearRegressor

In [22]:
kmer_dfs = []
for k in [1, 2, 3]:
    kmerreg = KmerLinearRegressor(kmer_length=k)
    kmer_df_k = kmerreg.count_kmers(result.index.values)
    kmer_df_k.columns = pd.MultiIndex.from_product([
        ["kmer_counts"],
        kmer_df_k.columns
    ])
    kmer_dfs.append(kmer_df_k)
    print(f"k={k} done")
result = pd.concat([result] + kmer_dfs, axis=1)

k=1 done
k=2 done
k=3 done


In [23]:
result["kmer_counts"]

,A,C,G,T,AA,AC,AG,AT,CA,CC,...,TCG,TCT,TGA,TGC,TGG,TGT,TTA,TTC,TTG,TTT
seq,,,,,,,,,,,,,,,,,,,,,
TTTTCTAATCTTTTCCCCCCTTTCCTCTTTTATTTACTTGTCTTGTATCC,5,15,2,28,1,1,0,3,0,7,...,0,4,0,0,0,2,2,3,2,8
ATTTTAATTTTTTTATTTGGTTTGATTATTAGTTTTATTGTTTTGATTTT,9,0,6,35,1,0,1,7,0,0,...,0,0,2,0,1,1,5,0,4,15
CCTCCCCCTCGTATTTCTCCTCCCTAGGTCTTCCTGTGTCGTCCCCACGC,3,24,7,16,0,1,1,1,1,12,...,2,2,0,0,0,2,0,2,0,1
AAGAATAATAAGAGAGGGGGTAAAAATATTAAGAGGACGAACATAAAAAG,28,2,13,7,14,2,7,5,1,0,...,0,0,0,0,0,0,1,0,0,0
CGCGAAGAAATGCGTCGAGGTGAGAGTGTACGCTGACGCACAAGCAACAG,16,11,17,6,5,4,6,1,4,0,...,1,0,2,1,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GCGATGAGGGGGGGGAGGGGGGGTGGAGGGGTGCTGGCGGTAGGGGGGGA,6,3,36,5,0,0,4,1,0,0,...,0,0,1,1,2,0,0,0,0,0
ATTTATATATGTATATAAAAAATTGAGATATAACAAAGGTGCATTTAATT,23,2,6,19,9,1,2,11,2,0,...,0,0,1,1,0,1,2,0,1,2
GTAGCTGGAGAGCTTGTGGGAGAGGATGATGCTGGTGCTTGTTTTGGGAG,8,4,23,15,0,0,6,2,0,0,...,0,0,1,2,4,2,0,0,3,2


In [24]:
result.to_parquet(f"embeddings_mapperless_{utr_type}_dirichlet.parquet")

---